# Timestamp Analysis of HVAC Data

In [1]:
import pandas as pd
import numpy as np

### Example on Table 1

In [2]:
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "timeseries_table_timeseries_table " + "(1)" + ".csv"

# Load the latest version
df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "lsobieski/processed-thermostat-data",
  file_path,
)

df.head()

,Timestamp,Temperature,Setpoint,FanState,OutputState,RunningMode
0,2025-02-05 15:53:08,NaN,0.0,0,0,0
1,2025-02-05 15:53:10,NaN,0.0,0,0,0
2,2025-02-05 16:05:10,NaN,23.0,0,0,0
3,2025-02-05 16:39:33,NaN,23.0,0,0,0
4,2025-02-05 16:39:34,NaN,23.0,0,0,0


In [3]:
df["TimestampAnalysis"] = pd.to_datetime(df["Timestamp"])
# change resampling letter to change timeframe (daily or monthly are the other two I've considered)
# choose thursday - why? we don't care about vacations necessarily if we have 3 months of good data with 1 week of vacation...
# this should minimize the impact vacations have on breaking up streaks
interval_counts = (
    df
    .set_index("TimestampAnalysis")
    .resample("W-THU")
    .size()
)

#change this number to change the number of the intervals we are averaging
top_intervals = interval_counts.sort_values(ascending=False).head(8)
avg_top = top_intervals.mean()

#change this to change minimum required for a week to be included in interval
bound = 0.35 * avg_top

boolean_intervals = interval_counts > bound

# https://joshdevlin.com/blog/calculate-streaks-in-pandas/
streak_start = boolean_intervals.ne(boolean_intervals.shift())

group_id = streak_start.cumsum()

true_groups = interval_counts[boolean_intervals].groupby(group_id)
longest_stretch = None
longest_length = 0

for _, group in true_groups:
    current_length = len(group)

    if current_length > longest_length:
        longest_length = current_length
        longest_stretch = group

print(longest_stretch)
print("Start Date:", longest_stretch.index[0].date(), "\nEnd Date:", longest_stretch.index[-1].date())

TimestampAnalysis
2025-12-04     8691
2025-12-11    10049
2025-12-18     5212
2025-12-25     3658
2026-01-01     4153
2026-01-08     5778
2026-01-15    11852
2026-01-22    12148
2026-01-29     8553
2026-02-05     8513
2026-02-12     5745
2026-02-19     3310
Freq: W-THU, dtype: int64
Start Date: 2025-12-04 
End Date: 2026-02-19


### Loopable Timestamp Analysis

In [7]:
for i in range(1, 101):
    # Set the path to the file you'd like to load
    file_path = "timeseries_table_timeseries_table (" + str(i) + ").csv"
    
    df = kagglehub.dataset_load(
      KaggleDatasetAdapter.PANDAS,
      "lsobieski/processed-thermostat-data",
      file_path,
    )

    df["TimestampAnalysis"] = pd.to_datetime(df["Timestamp"])
    # change resampling letter to change timeframe (daily or monthly are the other two I've considered)
    # choose thursday - why? we don't care about vacations necessarily if we have 3 months of good data with 1 week of vacation...
    # this should minimize the impact vacations have on breaking up streaks
    interval_counts = (
        df
        .set_index("TimestampAnalysis")
        .resample("W-THU")
        .size()
    )
    
    #change this number to change the number of the intervals we are averaging
    top_intervals = interval_counts.sort_values(ascending=False).head(8)
    avg_top = top_intervals.mean()
    
    #change this to change minimum required for a week to be included in interval
    bound = 0.35 * avg_top
    
    boolean_intervals = interval_counts > bound
    
    streak_start = boolean_intervals.ne(boolean_intervals.shift())
    
    group_id = streak_start.cumsum()
    
    true_groups = interval_counts[boolean_intervals].groupby(group_id)
    longest_stretch = None
    longest_length = 0
    
    for _, group in true_groups:
        current_length = len(group)
    
        if current_length > longest_length:
            longest_length = current_length
            longest_stretch = group

    print(str(i) + ":",  longest_stretch.index[0].date(), "-", longest_stretch.index[-1].date())
        

1: 2025-12-04 - 2026-02-19
2: 2025-11-27 - 2026-02-19
3: 2025-12-11 - 2026-02-12
4: 2025-11-27 - 2026-02-19
5: 2025-07-24 - 2025-08-28
6: 2025-10-02 - 2026-02-19
7: 2025-10-30 - 2026-02-12
8: 2025-12-04 - 2026-02-12
9: 2025-11-27 - 2026-02-19
10: 2025-04-17 - 2025-10-09
11: 2025-11-06 - 2026-02-12
12: 2025-08-14 - 2026-01-01
13: 2025-11-27 - 2026-02-12
14: 2025-10-09 - 2026-02-19
15: 2025-11-27 - 2026-02-19
16: 2025-07-03 - 2025-10-09
17: 2025-06-19 - 2025-10-23
18: 2025-05-22 - 2025-09-25
19: 2025-10-23 - 2026-02-19
20: 2025-11-06 - 2026-02-19
21: 2025-11-06 - 2026-02-12
22: 2025-12-25 - 2026-02-12
23: 2025-02-27 - 2025-07-03
24: 2025-11-13 - 2026-02-12
25: 2025-05-15 - 2025-10-16
26: 2025-11-06 - 2026-02-19
27: 2025-11-06 - 2026-02-19
28: 2025-08-14 - 2025-11-13
29: 2025-06-05 - 2025-11-13
30: 2025-11-27 - 2026-02-19
31: 2025-11-27 - 2026-02-19
32: 2025-11-06 - 2026-02-19
33: 2025-08-14 - 2026-02-19
34: 2025-06-05 - 2025-10-09
35: 2025-06-26 - 2025-10-09
36: 2025-11-13 - 2026-01-29
3